# cell3d inference — submit (isotropic lineage model)

Submission notebook for `IsotropicLineageNet`: native-resolution volumes, 4-view
TTA, an event ILP with per-node event costs and a drift-corrected distance term,
then linefit smoothing.

**Shipped model: `P-l50-s2`** (2026-09-24, EXP-20260924-01). It is the round-1
noisy-student winner: trained from scratch on GT plus W-link-s2's pipeline
pseudo-labels (lambda 0.5). It has 2.47M parameters and was trained on 165 train
movies; val and the 4 pinned movies were held out.
- Retuned recipe (2026-09-24, EXP-20260924-05): `disappearance_weight 6` and
  drift-compensated smoothing (the smoother runs in the stage's frame), every
  other knob at the `IsotropicConfig` default. This is the exact config its val
  numbers were measured with.
- Val (30 movies), 2-fold CV official adjusted edge Jaccard: 0.9298 (0.9309 at
  the chosen recipe); it was 0.9225 on the previous recipe (disappearance 4,
  plain smoothing), against the previous teacher's 0.9148.

Separate from `inference.ipynb` in this directory, which runs the shipped
three-model `TemporalUNet3D` pipeline. Neither notebook affects the other; they
share only the submission writer.


## 1. Bootstrap

The only step that cannot live in the package, because it installs the package.
Everything after it is a normal import.

Off Kaggle this just puts `src/` on the path.

In [1]:
import os
# Before torch initialises CUDA: a 4-member ensemble's dense pass fragmented a 16 GB V100 into an OOM (7.4 GiB
# allocated + 5.1 GiB reserved-but-free, 2026-09-28); expandable segments avoid that, numerics unchanged.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import subprocess
import sys
import zipfile
from pathlib import Path

IS_KAGGLE = Path("/kaggle/working").exists()

if IS_KAGGLE:
    # Both mount conventions, because Kaggle uses both and neither is reliable.
    _CANDIDATES = [
        Path("/kaggle/input/datasets/jamalsaeedi/cell3d-isotropic-lineage-c"),
        Path("/kaggle/input/cell3d-isotropic-lineage-c"),
        Path("/kaggle/input/datasets/jamalsaeedi/cell3d-inference-artifacts"),
        Path("/kaggle/input/cell3d-inference-artifacts"),
    ]
    ARTIFACTS_DIR = next((c for c in _CANDIDATES if c.exists()), _CANDIDATES[0])

    # A directory upload can land as `<name>.zip`; handle either.
    _src = ARTIFACTS_DIR / "src"
    if not _src.is_dir():
        _staging = Path("/kaggle/working/cell3d_artifacts")
        _src = _staging / "src"
        if not _src.exists():
            _src.mkdir(parents=True)
            with zipfile.ZipFile(ARTIFACTS_DIR / "src.zip") as bundle:
                bundle.extractall(_src)

    _wheels = sorted(_src.glob("cell_3d_link-*.whl"))
    assert _wheels, f"no cell_3d_link wheel in {_src}"
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", str(_wheels[-1])],
        check=True,
    )
    print(f"installed {_wheels[-1].name}")
    REPO_SRC = None
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    ARTIFACTS_DIR = REPO_ROOT / "artifacts" / "kaggle_inference_pack"
    REPO_SRC = REPO_ROOT / "src"

print(f"IS_KAGGLE={IS_KAGGLE}  ARTIFACTS_DIR={ARTIFACTS_DIR}")

Processing /kaggle/input/datasets/jamalsaeedi/cell3d-isotropic-lineage-c/src/cell_3d_link-0.1.0-py3-none-any.whl
installed cell_3d_link-0.1.0-py3-none-any.whl
IS_KAGGLE=True  ARTIFACTS_DIR=/kaggle/input/datasets/jamalsaeedi/cell3d-isotropic-lineage-c


## 2. Install the pinned wheels and locate the models

`bootstrap` is stdlib-only by design — it runs before numpy, torch, polars and
tracksdata are installed, so it cannot import them. It extracts `wheels/` and
`models/` (zip or directory), `pip install --no-index --no-deps` every wheel *by
file path*, and sets `CELL3D_MODELS_DIR` so `env.models_dir()` resolves once the
package is installed flat into site-packages.

**numpy and scipy are never reinstalled.** Both are already imported by the time
a cell runs, and reinstalling either in place corrupts the cached `numpy._core`
tree for everything loaded afterwards — a real run died with `cannot import name
'_center' from 'numpy._core.umath'`. This pipeline builds its candidate graph
with `scipy.spatial.cKDTree`, so that failure would land directly on it.

In [2]:
from cell_3d_link.isotropic.kaggle import bootstrap

environment = bootstrap(ARTIFACTS_DIR, repo_src=REPO_SRC)
print(environment.describe())

Processing /kaggle/input/datasets/jamalsaeedi/cell3d-isotropic-lineage-c/wheels/annotated_doc-0.0.4-py3-none-any.whl
annotated-doc is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Processing /kaggle/input/datasets/jamalsaeedi/cell3d-isotropic-lineage-c/wheels/annotated_types-0.7.0-py3-none-any.whl
annotated-types is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Processing /kaggle/input/datasets/jamalsaeedi/cell3d-isotropic-lineage-c/wheels/bidict-0.23.1-py3-none-any.whl
Processing /kaggle/input/datasets/jamalsaeedi/cell3d-isotropic-lineage-c/wheels/blosc2-4.7.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
  Attempting uninstall: blosc2
    Found existing installation: blosc2 4.1.2
    Uninstalling blosc2-4.1.2:
      Successfully uninstalled blosc2-4.1.2
Processing /kaggle/input/datasets/jamalsaeedi/cell3d-isotropic-li

## 3. Configuration

The shipped recipe is `IsotropicConfig`'s defaults plus the knobs the model's own
val retune moved (`scripts/local/retune_state.py`). Solver costs are the model's
negative log-likelihoods: edge `-log P(parent)`, appearance `-log P(no parent)`,
node `-(centre_logit - node_logit_bias)`. `node_logit_bias` and
`disappearance_weight` are the node-count levers. The metric charges
over-prediction on nodes.

The checkpoint's sha256 is pinned, so a stale or wrong model fails here, not after
the run.


In [3]:
from cell_3d_link.isotropic.checkpoint import checkpoint_summary, resolve_checkpoint
from cell_3d_link.isotropic.config import IsotropicConfig

CHECKPOINT_RUN = "A2-v10-e48-noisy-lc-s21-last"
# Ensemble members, run -> sha256 of its best.pt (association mode: each member samples its own features
# and scores the shared candidates with its own head; the parent distributions are averaged). Empty = one
# model. RESCORER names the trees trained on THIS recipe's val decode (the ensemble's own, for an ensemble).
ENSEMBLE_RUNS = {"B2-v10-e48-noisy-lc-s22-last": "62604e555e7ca3a354feba3f39c154045562b20115935f6fafdad8e69142c33f", "CX-deep-lc-s3": "ad700592cbafa94b55018619cb4a334729f8011a24d3558da78fbc3a429c3428", "DX-widedeep-e48-lc-s4-last": "b3adbdaa0f37d0137a50821368f6f29fd6596b386086bc34a2e8ab7c8f030010", "EX-ms-lc-s5-last": "8545492a809169bfa33486c717e39ed43940fda51e802d9ef536b33c05e455cf", "FX-ms-lc-s6-last": "e51785fa7ef84a6378273e2cfc8999815430584256a02744bcab7c253d761e6f"}
RESCORER = CHECKPOINT_RUN
# Member-own re-scorers (MRES, ens_mres.sh): one per model, primary first, each trained on THAT member alone
# on the ensemble\'s node set (gbt_link_lab export --train-states FUSE-<tag>-m<k>-val). They REPLACE RESCORER.
# Empty = off (RESCORER on the fused terms).
MEMBER_RESCORERS = ["MRES-S9X-CD333-m0", "MRES-S9X-CD333-m1", "MRES-S9X-CD333-m2", "MRES-S9X-CD333-m3", "MRES-S9X-CD333-m4", "MRES-S9X-CD333-m5"]
# Stacked re-scorer (ens_fusion_lab.py stack, 2026-09-28): ONE tree ensemble on the members' averaged terms plus
# every member's own view of each edge (edge_rescorer.stack_features). It REPLACES RESCORER and MEMBER_RESCORERS.
# None = off.
STACK_RESCORER = None

# The measured recipe, and nothing else: IsotropicConfig's defaults (4-view TTA,
# threshold 0.5, drift-corrected distance x0.2, linefit smoothing, repair off)
# plus what R3-ft24-noisy-lc-s1's val retune moved: disappearance 10 (best of grid;
# the 2-fold CV picked d10 and d12) and drift-compensated smoothing. Every other value
# here is what its val score was measured with (EXP-20260925-01, CV 0.9346) -- do not
# add overrides that were not measured.
# Edge re-scorer (EXP-20260925-12): a gradient-boosted re-ranking of each target's
# candidate parents, trained on lc-s1's held-out val decode (30 movies), shipped inside
# the package as numpy/numba trees; the model's null mass is kept. Its retune picked
# disappearance 12 (CV d12 / d12+bias 0.5). Val 2-fold CV 0.9406 vs 0.9351 (+0.0054
# [+0.0012, +0.0100], out-of-fold trees); pinned 4 0.9299 vs 0.9255 (trees never saw
# them); val + pinned (34) +0.0051 [+0.0011, +0.0096]. ~3 s per movie on 4 CPU threads.
# Precision (EXP-20260925-13): amp_dtype "auto" = fp16 on this T4 (no native bf16; the
# emulated bf16 decode was ~2.5x slower and two hidden-test runs missed the 12 h limit).
# fp16 on a V100 vs bf16 on the 3090: val -0.0004 [-0.0012, +0.0001], pinned 4 identical.
# The shard workers also degrade to 1 TTA view if the pace would miss the deadline.
from dataclasses import replace
from pathlib import Path

config = IsotropicConfig(checkpoint=resolve_checkpoint(CHECKPOINT_RUN))
config = config.with_solver(disappearance_weight=12.0, appearance_bias=0.5, association_temperature=0.9, edge_distance_weight=0.25, node_logit_bias=0.08)
import cell_3d_link.isotropic as _iso
config = replace(config, edge_rescorer=Path(_iso.__file__).parent / "rescorers" / f"{RESCORER}.npz")
# (an ensemble with member-own or stacked trees replaces this below: RESCORER then need not exist)
assert config.edge_rescorer.is_file() or MEMBER_RESCORERS or STACK_RESCORER, f"no edge re-scorer at {config.edge_rescorer}"
config = replace(config, smooth_drift_compensated=True)
# z-axis line-fit window (docs/TRACKING_2026-09-26.md section 5.1); y / x keep the default window 2.
# None = the lateral window, the shipped behaviour; SMOOTH_ENDS "extrapolate" is the shipped behaviour.
# Only applied when moved, so an older wheel without these fields still runs this cell unchanged.
SMOOTH_WINDOW_Z = None
SMOOTH_ENDS = "extrapolate"
if SMOOTH_WINDOW_Z is not None or SMOOTH_ENDS != "extrapolate":
    config = config.with_smoothing(window_z=SMOOTH_WINDOW_Z, ends=SMOOTH_ENDS)
# Detection overrides (DetectionConfig field -> value), e.g. input preprocessing / photometric TTA measured on
# split-A val; empty = the shipped detection. Only applied when non-empty (an older wheel runs this cell unchanged).
DETECTION_OVERRIDES = {'ensemble_weights': (1, 1, 1, 1, 0, 0), 'member_tta_views': (4, 4, 4, 3, 3, 3), 'encoder_cache': True}
if DETECTION_OVERRIDES:
    config = config.with_detection(**DETECTION_OVERRIDES)

EXPECTED_SHA256 = "af6ec6fd722b83b48579ecd2b67927a3f0d332017d05cf4f9100d594f6e67d03"

import hashlib
_sha = hashlib.sha256(config.checkpoint.read_bytes()).hexdigest()
assert _sha == EXPECTED_SHA256, f"checkpoint sha256 {_sha} != pinned {EXPECTED_SHA256}"
if ENSEMBLE_RUNS:
    _extra = {}
    if STACK_RESCORER:
        assert not MEMBER_RESCORERS, "STACK_RESCORER replaces MEMBER_RESCORERS"
        _extra = dict(edge_rescorer=None, stack_rescorer=Path(_iso.__file__).parent / "rescorers" / f"{STACK_RESCORER}.npz")
        assert _extra["stack_rescorer"].is_file(), f"no stacked re-scorer at {_extra['stack_rescorer']}"
    elif MEMBER_RESCORERS:  # one replace: IsotropicConfig validates every intermediate config
        assert len(MEMBER_RESCORERS) == 1 + len(ENSEMBLE_RUNS), "one member re-scorer per model, primary first"
        _extra = dict(edge_rescorer=None, ensemble_rescorers=tuple(
            Path(_iso.__file__).parent / "rescorers" / f"{r}.npz" for r in MEMBER_RESCORERS))
    config = replace(config, ensemble_checkpoints=tuple(resolve_checkpoint(r) for r in ENSEMBLE_RUNS),
                     ensemble_mode="association", **_extra)
    for _p in getattr(config, "ensemble_rescorers", ()):
        assert _p.is_file(), f"no member re-scorer at {_p}"
    for _path, (_run, _want) in zip(config.ensemble_checkpoints, ENSEMBLE_RUNS.items()):
        _got = hashlib.sha256(_path.read_bytes()).hexdigest()
        assert _got == _want, f"ensemble member {_run} sha256 {_got} != pinned {_want}"

summary = checkpoint_summary(config.checkpoint)
print(f"checkpoint   {config.checkpoint}  (sha256 ok)")
print(f"  run        {summary['run_name']}  step {summary['step']}")
print(f"  split      {summary['split_digest']}")
print(f"  model      {summary['model']}")
d, s = config.detection, config.solver
print(f"detection    threshold {d.threshold}, tta_views {d.tta_views}"
      f"{', overrides ' + repr(DETECTION_OVERRIDES) if DETECTION_OVERRIDES else ''}")
print(f"candidates   {config.candidates.radius_um} um, k={config.candidates.max_per_node}")
print(f"solver       node_bias {s.node_logit_bias}, disappear {s.disappearance_weight}, "
      f"distance {s.edge_distance_mode} x{s.edge_distance_weight}, div_bias {s.division_bias}, "
      f"appearance_bias {s.appearance_bias}")
print(f"ensemble     {', '.join(ENSEMBLE_RUNS) if ENSEMBLE_RUNS else 'off'}"
      f"{' (association, own heads)' if ENSEMBLE_RUNS else ''}")
_mres = getattr(config, "ensemble_rescorers", ())
_stack = getattr(config, "stack_rescorer", None)
print(f"rescorer     {'stack ' + _stack.name if _stack else ('members ' + ', '.join(p.name for p in _mres) if _mres else (config.edge_rescorer.name if config.edge_rescorer else 'off'))}")
print(f"smoothing    {config.smoothing.mode if config.smoothing.enabled else 'off'}"
      f"{', drift-compensated' if config.smooth_drift_compensated else ''}"
      f", weight {config.smoothing.weight}, window {config.smoothing.window}"
      f", window_z {getattr(config.smoothing, 'window_z', None)}"
      f", ends {getattr(config.smoothing, 'ends', 'extrapolate')}")
print(f"repair       {'ON' if config.repair.enabled else 'off'}")


checkpoint   /kaggle/input/datasets/jamalsaeedi/cell3d-isotropic-lineage-c/models/isotropic_lineage/A2-v10-e48-noisy-lc-s21-last/best.pt  (sha256 ok)
  run        A2-v10-e48-noisy-lc-s21  step 7424
  split      d04c9617bf8b82d7
  model      {'stem_channels': 8, 'feature_channels': 48, 'association_hidden': 192, 'association_blocks': 3, 'input_normalization': 'none', 'feature_dropout': 0.2}
detection    threshold 0.5, tta_views 4, overrides {'ensemble_weights': (1, 1, 1, 1, 0, 0), 'member_tta_views': (4, 4, 4, 3, 3, 3), 'encoder_cache': True}
candidates   20.0 um, k=4
solver       node_bias 0.08, disappear 12.0, distance drift x0.25, div_bias 0.0, appearance_bias 0.5
ensemble     B2-v10-e48-noisy-lc-s22-last, CX-deep-lc-s3, DX-widedeep-e48-lc-s4-last, EX-ms-lc-s5-last, FX-ms-lc-s6-last (association, own heads)
rescorer     members MRES-S9X-CD333-m0.npz, MRES-S9X-CD333-m1.npz, MRES-S9X-CD333-m2.npz, MRES-S9X-CD333-m3.npz, MRES-S9X-CD333-m4.npz, MRES-S9X-CD333-m5.npz
smoothing    linefit,

## 4. Data and run plan

`plan_run` resolves device, shard count and the ILP time guard from what is
actually visible.

**Dual-GPU sharding on Kaggle.** One OS subprocess per visible GPU — subprocesses
rather than `multiprocessing`, because a notebook cell is an unguarded
`__main__` where `spawn` fails outright — with a filesystem `os.rename` claim
queue so whichever GPU is faster simply claims more movies. A 2×T4 kernel gets
2 shards; a single-GPU kernel falls back to the single-process path with no
subprocess overhead. Sharding is also capped by the movie count.

**The ILP is the only unbounded stage.** Measured on a full 100-frame movie:
8 s decode, 0.5 s association, **193 s of SCIP**. Each movie's cap is
`reserve × remaining_session / remaining_movies` against *actual* elapsed time,
so a hidden run with many more movies than the visible sample tightens every
later cap automatically. No movie count is assumed anywhere.

In [4]:
import torch

from cell_3d_link import env
from cell_3d_link.isotropic.kaggle import plan_run

assert env.is_kaggle() == IS_KAGGLE
data_dir = env.data_root("test")
working_dir = env.working_dir()
stems = sorted(p.stem for p in data_dir.glob("*.zarr"))
print(f"data_dir    {data_dir}")
print(f"working_dir {working_dir}")
print(f"movies      {stems}")
print()

plan = plan_run(len(stems), session_hours=12.0, ilp_reserve_fraction=0.3)
print(plan.describe())
if not torch.cuda.is_available():
    print("\nWARNING: no GPU — this pipeline will not finish on CPU in reasonable time.")

data_dir    /kaggle/input/competitions/biohub-cell-tracking-during-development/test
working_dir /kaggle/working
movies      ['44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1']



/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)


device cuda  (2 GPU(s))
  GPU 0: Tesla T4
  GPU 1: Tesla T4
  autocast     torch.float16  (Turing and older have no bf16)
  sharding     2 process(es)
  movies       4
  ILP guard    <= 30% of remaining session / remaining movies, deadline 12.0 h


## 5. Run

`run_pipeline` predicts every movie, applies the repair ladder if enabled,
enforces the submission's structural contract, writes the CSV and validates it —
raising on a schema change, non-contiguous ids, a dangling or non-consecutive
edge, in-degree above 1 or out-degree above 2.

In [5]:
from cell_3d_link.isotropic.pipeline import run_pipeline

submission_path = working_dir / "submission.csv"
predictions_dir = working_dir / "predictions"
timing = {}

submission_path = run_pipeline(
    data_dir,
    predictions_dir,
    submission_path,
    config=config,
    device=plan.device,
    num_shards=plan.num_shards,
    session_deadline_seconds=plan.session_deadline_seconds,
    ilp_reserve_fraction=plan.ilp_reserve_fraction,
    timing=timing,
)
print(f"\nSubmission written and validated at: {submission_path}")
print(f"  {timing['nodes']} nodes, {timing['edges']} edges, {timing['divisions']} divisions")
print(f"  predict {timing['predict_seconds'] / 60:.1f} min, "
      f"total {timing['total_seconds'] / 60:.1f} min")


Submission written and validated at: /kaggle/working/submission.csv
  134991 nodes, 131881 edges, 59 divisions
  predict 14.8 min, total 15.0 min


## 6. Check what was written

Reaching this cell means the file is structurally valid — `run_pipeline` already
validated it. What is left is whether the *quantities* are sane.

The number to watch is `node_ratio`: predicted nodes over the movie's own
`estimated_number_of_nodes`. The metric charges over-prediction directly, and
`solver.node_logit_bias` is the lever if this is too high.

Per-movie repair counts are printed too, so a stage that fired unexpectedly
often is visible rather than buried.

In [6]:
import pandas as pd

from cell_3d_link.isotropic.volumes import estimated_node_count

df = pd.read_csv(submission_path)
print(df["row_type"].value_counts().to_string())
print(f"datasets: {sorted(df['dataset'].unique())}")

rows = []
for stem, group in df.groupby("dataset"):
    nodes = int((group["row_type"] == "node").sum())
    truth = estimated_node_count(data_dir / stem)
    rows.append({
        "dataset": stem,
        "nodes": nodes,
        "edges": int((group["row_type"] == "edge").sum()),
        "estimated": truth,
        "node_ratio": round(nodes / truth, 3) if truth else None,
    })
print()
print(pd.DataFrame(rows).to_string(index=False))

if config.repair.enabled:
    print("\nrepair counts per movie:")
    for stem, stat in timing["repair_stats"].items():
        fired = {k: v for k, v in stat.items()
                 if k.startswith(("gap_", "short_", "isolated_")) and v}
        print(f"  {stem}: {fired or 'nothing fired'}")

df.head(10)

row_type
node    134991
edge    131881
datasets: ['44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1']

      dataset  nodes  edges estimated node_ratio
44b6_0113de3b  26870  26154      None       None
44b6_0b24845f  26830  26181      None       None
6bba_05b6850b   5846   5719      None       None
6bba_05db0fb1  75445  73827      None       None


,id,dataset,row_type,node_id,t,z,y,x,source_id,target_id
0,0,44b6_0113de3b,node,0,0,1,8,52,-1,-1
1,1,44b6_0113de3b,node,1,0,1,6,71,-1,-1
2,2,44b6_0113de3b,node,2,0,1,27,65,-1,-1
3,3,44b6_0113de3b,node,3,0,1,52,82,-1,-1
4,4,44b6_0113de3b,node,4,0,1,99,35,-1,-1
5,5,44b6_0113de3b,node,5,0,1,107,75,-1,-1
6,6,44b6_0113de3b,node,6,0,1,126,71,-1,-1
7,7,44b6_0113de3b,node,7,0,1,163,28,-1,-1
8,8,44b6_0113de3b,node,8,0,1,179,63,-1,-1
9,9,44b6_0113de3b,node,9,0,1,186,39,-1,-1
